In [0]:
%pip install faker

In [0]:
# =============================================================================
# Smart ERP – Silver Layer Simulation  (Databricks / Delta Lake)
# Catalog  : smart_odoo
# Targets  : silver.res_partner | silver.res_users | silver.res_groups
#
# IDEMPOTENCY GUARANTEES
# ──────────────────────
# ✅ Fixed seed     → same names / dates on every run (no random drift)
# ✅ MERGE on PK    → re-running never creates duplicate rows
# ✅ CREATE IF NOT EXISTS handled by Delta saveAsTable + MERGE logic
# ✅ Temp-view names are unique per table → no stale-view collisions
# ✅ dwh_loaded_at  → always updated to current_timestamp on each upsert
# ✅ No DROP / TRUNCATE → safe to re-run in production
#
# Rules
# ──────
#   partner_id  1 – 50   → Suppliers  (is_company=True, supplier_rank=1)
#   partner_id 51 – 250  → Customers  (mixed is_company,  customer_rank=1)
#   Date range           : 2025-01-01 → 2026-04-22
# =============================================================================

# ── 0. Imports ────────────────────────────────────────────────────────────────
import random
import string
import json
from datetime import datetime, timedelta, timezone

from faker import Faker
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, BooleanType, TimestampType,
)

# ── Fixed seeds → identical output on every run ───────────────────────────────
SEED = 42
Faker.seed(SEED)
random.seed(SEED)

fake  = Faker()
spark = SparkSession.builder.getOrCreate()

# ── 1. Constants & helpers ────────────────────────────────────────────────────
CATALOG    = "smart_odoo"
SCHEMA     = "silver"
SOURCE_DB  = "python_ingest"

CITIES     = ["Cairo", "Alexandria", "Giza", "Dubai", "Riyadh"]
DOMAINS    = ["gmail.com", "yahoo.com", "company.com"]
INDUSTRIES = [1, 2, 3, 4, 5, None]

START_DATE = datetime(2025, 1, 1)
END_DATE   = datetime(2026, 6, 15)

COUNTRY_LOOKUP = {
    "Cairo":      (20,  "Egypt",        None, None),
    "Alexandria": (20,  "Egypt",        None, None),
    "Giza":       (20,  "Egypt",        None, None),
    "Dubai":      (784, "UAE",          None, None),
    "Riyadh":     (682, "Saudi Arabia", None, None),
}

SALE_TEAMS = [
    (1, "Direct Sales"),
    (2, "Online Sales"),
    (3, "Field Sales"),
    (None, None),
]

COMPANIES = [
    (1, "Alpha Corp"),
    (2, "Beta Ltd"),
    (3, "Gamma Inc"),
]

GROUPS_DATA = [
    (1,  "Sales / User",         1, "Sales",     "Basic sales access"),
    (2,  "Sales / Manager",      1, "Sales",     "Full sales access"),
    (3,  "Inventory / User",     2, "Inventory", "Read inventory"),
    (4,  "Inventory / Manager",  2, "Inventory", "Full inventory access"),
    (5,  "Accounting / Billing", 3, "Finance",   "Billing only"),
    (6,  "Accounting / Manager", 3, "Finance",   "Full finance access"),
    (7,  "Purchase / User",      4, "Purchase",  "Purchase orders"),
    (8,  "Purchase / Manager",   4, "Purchase",  "Full purchase access"),
    (9,  "HR / Employee",        5, "HR",        "Employee self-service"),
    (10, "HR / Manager",         5, "HR",        "Full HR access"),
    (11, "Technical / Settings", 6, "Technical", "System settings"),
    (12, "Base / Internal User", 6, "Technical", "Internal user base"),
]


def _random_date() -> datetime:
    delta = (END_DATE - START_DATE).days
    return START_DATE + timedelta(days=random.randint(0, delta))


def _phone() -> str:
    return "01" + "".join(random.choices(string.digits, k=9))


def _email(name: str) -> str:
    base = name.replace(" ", "").lower()[:20]
    return f"{base}@{random.choice(DOMAINS)}"


# ── 2. Build res_partner rows ─────────────────────────────────────────────────
partner_schema = StructType([
    StructField("partner_id",              IntegerType(),  False),
    StructField("partner_name",            StringType(),   True),
    StructField("company_id",              IntegerType(),  True),
    StructField("company_name",            StringType(),   True),
    StructField("parent_id",               IntegerType(),  True),
    StructField("parent_name",             StringType(),   True),
    StructField("commercial_partner_id",   IntegerType(),  True),
    StructField("commercial_partner_name", StringType(),   True),
    StructField("country_id",              IntegerType(),  True),
    StructField("country_name",            StringType(),   True),
    StructField("state_id",                IntegerType(),  True),
    StructField("state_name",              StringType(),   True),
    StructField("industry_id",             IntegerType(),  True),
    StructField("is_company",              BooleanType(),  True),
    StructField("active",                  BooleanType(),  True),
    StructField("customer_rank",           IntegerType(),  True),
    StructField("supplier_rank",           IntegerType(),  True),
    StructField("email",                   StringType(),   True),
    StructField("phone",                   StringType(),   True),
    StructField("created_at",              TimestampType(), True),
    StructField("updated_at",              TimestampType(), True),
    StructField("dwh_loaded_at",           TimestampType(), True),
    StructField("dwh_source_db",           StringType(),   True),
])

now = datetime.now(timezone.utc).replace(tzinfo=None)   # naive UTC for Delta

partner_rows = []

# ── 2a. Suppliers (1-50) — always companies ───────────────────────────────────
for pid in range(1, 51):
    created = _random_date()
    name    = fake.company()
    city    = random.choice(CITIES)
    cid, cname, sid, sname = COUNTRY_LOOKUP[city]

    partner_rows.append((
        pid, name,
        pid, name,                          # company_id / company_name = itself
        None, None,                         # no parent
        pid, name,                          # commercial_partner = itself
        cid, cname,
        sid, sname,
        random.choice(INDUSTRIES),
        True,                               # is_company
        True,                               # active
        0,                                  # customer_rank
        1,                                  # supplier_rank ← SUPPLIER
        _email(name), _phone(),
        created, created,
        now, SOURCE_DB,
    ))

# ── 2b. Customers (51-250) — mixed ────────────────────────────────────────────
# Build a fast lookup dict so parent-name resolution is O(1), not O(n)
supplier_name_by_id = {r[0]: r[1] for r in partner_rows}

for pid in range(51, 251):
    created    = _random_date()
    is_company = random.choice([True, False])
    name       = fake.company() if is_company else fake.name()
    city       = random.choice(CITIES)
    cid, cname, sid, sname = COUNTRY_LOOKUP[city]

    parent_pid, parent_name = None, None
    comm_pid,   comm_name   = pid, name

    # 30 % of individual customers are linked to a supplier company
    if not is_company and random.random() < 0.3:
        parent_pid  = random.randint(1, 50)
        parent_name = supplier_name_by_id[parent_pid]
        comm_pid    = parent_pid
        comm_name   = parent_name

    partner_rows.append((
        pid, name,
        pid if is_company else (parent_pid or pid),
        name if is_company else (parent_name or name),
        parent_pid, parent_name,
        comm_pid, comm_name,
        cid, cname,
        sid, sname,
        random.choice(INDUSTRIES),
        is_company, True,
        1,                                  # customer_rank ← CUSTOMER
        0,
        _email(name), _phone(),
        created, created,
        now, SOURCE_DB,
    ))

df_partner = spark.createDataFrame(partner_rows, schema=partner_schema)

# ── 3. Build res_users rows ───────────────────────────────────────────────────
user_schema = StructType([
    StructField("user_id",        IntegerType(),  False),
    StructField("company_id",     IntegerType(),  True),
    StructField("company_name",   StringType(),   True),
    StructField("partner_id",     IntegerType(),  True),
    StructField("partner_name",   StringType(),   True),
    StructField("sale_team_id",   IntegerType(),  True),
    StructField("sale_team_name", StringType(),   True),
    StructField("login",          StringType(),   True),
    StructField("active",         BooleanType(),  True),
    StructField("share",          BooleanType(),  True),
    StructField("created_at",     TimestampType(), True),
    StructField("updated_at",     TimestampType(), True),
    StructField("dwh_loaded_at",  TimestampType(), True),
    StructField("dwh_source_db",  StringType(),   True),
])

# Customer rows only (index 50 onward in partner_rows = id 51-250)
customer_rows = [r for r in partner_rows if r[0] >= 51]

user_rows = []
for uid in range(1, 31):
    created         = _random_date()
    linked_partner  = customer_rows[uid - 1]        # deterministic pick, not random.choice
    pid, pname      = linked_partner[0], linked_partner[1]
    cid, cname      = COMPANIES[(uid - 1) % len(COMPANIES)]
    tid, tname      = SALE_TEAMS[(uid - 1) % len(SALE_TEAMS)]

    user_rows.append((
        uid,
        cid, cname,
        pid, pname,
        tid, tname,
        _email(pname),
        True,
        (uid % 2 == 0),                     # deterministic share flag
        created, created,
        now, SOURCE_DB,
    ))

df_users = spark.createDataFrame(user_rows, schema=user_schema)

# ── 4. Build res_groups rows ──────────────────────────────────────────────────
group_schema = StructType([
    StructField("group_id",       IntegerType(),  False),
    StructField("group_name",     StringType(),   True),
    StructField("privilege_id",   IntegerType(),  True),
    StructField("privilege_name", StringType(),   True),
    StructField("share",          BooleanType(),  True),
    StructField("group_comment",  StringType(),   True),
    StructField("created_at",     TimestampType(), True),
    StructField("updated_at",     TimestampType(), True),
    StructField("dwh_loaded_at",  TimestampType(), True),
    StructField("dwh_source_db",  StringType(),   True),
])

group_rows = []
for gid, gname, priv_id, priv_name, comment in GROUPS_DATA:
    created = _random_date()
    group_rows.append((
        gid, gname,
        priv_id, priv_name,
        False,
        comment,
        created, created,
        now, SOURCE_DB,
    ))

df_groups = spark.createDataFrame(group_rows, schema=group_schema)


# ── 5. Idempotent MERGE writer ────────────────────────────────────────────────
def upsert_silver(df, table: str, pk: str) -> None:
    """
    MERGE df into smart_odoo.silver.<table> on <pk>.

    Safe to call multiple times:
      - Creates the Delta table on first run (via saveAsTable).
      - On subsequent runs, MERGEs: updates matched rows, inserts new ones.
      - dwh_loaded_at is always refreshed to current_timestamp().
      - Temp view name is unique per table to avoid cross-table collisions.
    """
    full_table = f"{CATALOG}.{SCHEMA}.{table}"
    view_name  = f"_sim_src_{table}"           # unique name per table

    # ── Ensure table exists (no-op if already present) ────────────────────────
    # saveAsTable with "errorifexists" would fail on re-run, so we use
    # a MERGE-only path once the table is confirmed to exist.
    table_exists = spark.catalog.tableExists(full_table)

    if not table_exists:
        # First ever run: write directly and create the Delta table
        (df.write
           .format("delta")
           .mode("errorifexists")             # hard-fail if race condition
           .option("overwriteSchema", "false")
           .saveAsTable(full_table))
        print(f"[created]  {full_table}  →  {df.count()} rows written")
        return

    # ── Subsequent runs: MERGE on PK ─────────────────────────────────────────
    df.createOrReplaceTempView(view_name)

    # Build SET clause for all columns except the PK
    # dwh_loaded_at is handled separately via current_timestamp()
    other_cols = [c for c in df.columns if c not in (pk, "dwh_loaded_at")]
    set_parts  = [f"t.{c} = s.{c}" for c in other_cols]
    set_parts.append("t.dwh_loaded_at = current_timestamp()")
    set_clause = ",\n        ".join(set_parts)

    spark.sql(f"""
        MERGE INTO {full_table} AS t
        USING {view_name}       AS s
        ON t.{pk} = s.{pk}

        WHEN MATCHED THEN UPDATE SET
            {set_clause}

        WHEN NOT MATCHED THEN INSERT *
    """)

    # Drop the temp view immediately to keep the session clean
    spark.catalog.dropTempView(view_name)

    print(f"[merged]   {full_table}  →  upserted {df.count()} rows")


# ── 6. Runtime assertions (catch issues before touching Delta) ────────────────
def _assert_data():
    partner_ids = [r[0] for r in partner_rows]
    assert len(partner_ids) == len(set(partner_ids)), \
        "❌ Duplicate partner_ids detected"

    user_ids = [r[0] for r in user_rows]
    assert len(user_ids) == len(set(user_ids)), \
        "❌ Duplicate user_ids detected"

    group_ids = [r[0] for r in group_rows]
    assert len(group_ids) == len(set(group_ids)), \
        "❌ Duplicate group_ids detected"

    suppliers = [r for r in partner_rows if r[0] <= 50]
    customers = [r for r in partner_rows if r[0] >= 51]

    assert all(r[16] == 1 and r[15] == 0 and r[13] is True for r in suppliers), \
        "❌ Supplier rank / is_company rule violated"
    assert all(r[15] == 1 and r[16] == 0 for r in customers), \
        "❌ Customer rank rule violated"

    for r in partner_rows:
        d = r[19]
        assert START_DATE <= d <= END_DATE, \
            f"❌ Date out of range: {d}  (partner_id={r[0]})"

    print("✅  All pre-flight assertions passed")

_assert_data()

# ── 7. Execute ────────────────────────────────────────────────────────────────
spark.sql(f"USE CATALOG {CATALOG}")

upsert_silver(df_partner, "res_partner", "partner_id")
upsert_silver(df_users,   "res_users",   "user_id")
upsert_silver(df_groups,  "res_groups",  "group_id")

print("\n✅  Simulation complete – smart_odoo.silver populated.")
print(f"   res_partner : {df_partner.count()} rows  "
      f"(1-50 suppliers | 51-250 customers)")
print(f"   res_users   : {df_users.count()} rows")
print(f"   res_groups  : {df_groups.count()} rows")